In [10]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gdown
import warnings
warnings.filterwarnings('ignore')

In [11]:
# Load the data
!pip -q install gdown

file_id = "1Qeoeg1HTnwLnF9dsbq2cPVZKZdi3dqpP"
output_file = "data.csv"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output_file, quiet=False)

df = pd.read_csv(output_file)
print(df.shape)
df.head()

Downloading...
From: https://drive.google.com/uc?id=1Qeoeg1HTnwLnF9dsbq2cPVZKZdi3dqpP
To: /content/data.csv
100%|██████████| 568k/568k [00:00<00:00, 89.3MB/s]

(12165, 11)


,dteday,hum,weathersit,holiday,season,atemp,temp,hr,casual,registered,cnt
0,2011-12-09,0.62,1,0,4,0.3485,0.36,16,24,226,250
1,2012-06-17,0.64,1,0,2,0.5152,0.54,4,2,16,18
2,2011-06-15,0.53,1,0,2,0.6212,0.62,23,17,90,107
3,2012-03-31,0.87,2,0,2,0.3485,0.36,8,19,126,145
4,2012-07-31,0.55,1,0,3,0.6970,0.76,18,99,758,857


## Bike Sharing Demand Prediction (Regression)

### Business Problem Understanding
Bike sharing operator membutuhkan prediksi jumlah penyewaan sepeda (`cnt`) agar alokasi sepeda dan sumber daya operasional lebih tepat.

Problem statement:
- Terjadi mismatch antara supply dan demand pada jam/jadwal tertentu.
- Dampak: kehilangan potensi transaksi saat demand tinggi dan inefisiensi saat demand rendah.

Project objective:
- Membangun model **regresi** untuk memprediksi `cnt`.
- Fokus evaluasi pada error prediksi (MAE dan RMSE) agar mudah diterjemahkan ke kebutuhan operasional.

### Data Understanding
Pada tahap ini dilakukan pengecekan struktur data, tipe data, missing value, duplikasi, dan statistik deskriptif awal untuk memastikan data siap diproses ke tahap modeling regresi.

In [12]:
# Additional imports for regression workflow
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import pickle

In [13]:
# Basic data checks
display(df.head())
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nDescriptive stats:')
display(df.describe(include='all').T)

,dteday,hum,weathersit,holiday,season,atemp,temp,hr,casual,registered,cnt
0,2011-12-09,0.62,1,0,4,0.3485,0.36,16,24,226,250
1,2012-06-17,0.64,1,0,2,0.5152,0.54,4,2,16,18
2,2011-06-15,0.53,1,0,2,0.6212,0.62,23,17,90,107
3,2012-03-31,0.87,2,0,2,0.3485,0.36,8,19,126,145
4,2012-07-31,0.55,1,0,3,0.6970,0.76,18,99,758,857


Shape: (12165, 11)

Data types:
dteday         object
hum           float64
weathersit      int64
holiday         int64
season          int64
atemp         float64
temp          float64
hr              int64
casual          int64
registered      int64
cnt             int64
dtype: object

Missing values:
dteday        0
hum           0
weathersit    0
holiday       0
season        0
atemp         0
temp          0
hr            0
casual        0
registered    0
cnt           0
dtype: int64

Duplicate rows: 0

Descriptive stats:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
dteday,12165,731,2012-03-02,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hum,12165.0,NaN,NaN,NaN,0.625451,0.192102,0.0,0.47,0.62,0.78,1.0
weathersit,12165.0,NaN,NaN,NaN,1.416934,0.635937,1.0,1.0,1.0,2.0,4.0
holiday,12165.0,NaN,NaN,NaN,0.029758,0.169925,0.0,0.0,0.0,0.0,1.0
season,12165.0,NaN,NaN,NaN,2.488615,1.106157,1.0,2.0,2.0,3.0,4.0
atemp,12165.0,NaN,NaN,NaN,0.476996,0.171857,0.0,0.3333,0.4848,0.6212,1.0
temp,12165.0,NaN,NaN,NaN,0.498185,0.192492,0.02,0.34,0.5,0.66,1.0
hr,12165.0,NaN,NaN,NaN,11.51977,6.931872,0.0,6.0,12.0,18.0,23.0
casual,12165.0,NaN,NaN,NaN,35.834443,49.489286,0.0,4.0,17.0,49.0,362.0
registered,12165.0,NaN,NaN,NaN,153.43658,151.046123,0.0,34.0,115.0,220.0,876.0


### Data Preprocessing
Tahap ini menyiapkan fitur untuk model regresi, termasuk konversi tanggal dan pemisahan train-test.

In [14]:
# Feature engineering and train-test split
data = df.copy()

data['dteday'] = pd.to_datetime(data['dteday'])
data['year'] = data['dteday'].dt.year
data['month'] = data['dteday'].dt.month
data['dayofweek'] = data['dteday'].dt.dayofweek

target_col = 'cnt'
feature_cols = [c for c in data.columns if c not in [target_col, 'dteday']]

X = data[feature_cols]
y = data[target_col]

categorical_features = ['season', 'weathersit', 'holiday', 'hr', 'year', 'month', 'dayofweek']
categorical_features = [c for c in categorical_features if c in X.columns]
numeric_features = [c for c in X.columns if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Train shape: (9732, 12)
Test shape : (2433, 12)
Numeric features: ['hum', 'atemp', 'temp', 'casual', 'registered']
Categorical features: ['season', 'weathersit', 'holiday', 'hr', 'year', 'month', 'dayofweek']


### Modeling (Regression)
Model yang dibandingkan:
- Linear Regression (baseline)
- Random Forest Regressor
- Gradient Boosting Regressor

In [ ]:
# Train and compare multiple regression models
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(random_state=42, n_estimators=250, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

    cv_scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={
            'mae': 'neg_mean_absolute_error',
            'rmse': 'neg_root_mean_squared_error',
            'r2': 'r2'
        },
        n_jobs=-1
    )

    pipe.fit(X_train, y_train)
    y_pred_test = pipe.predict(X_test)

    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_r2 = r2_score(y_test, y_pred_test)

    rows.append({
        'model': name,
        'cv_mae': -cv_scores['test_mae'].mean(),
        'cv_rmse': -cv_scores['test_rmse'].mean(),
        'cv_r2': cv_scores['test_r2'].mean(),
        'test_mae': test_mae,
        'test_rmse': test_rmse,
        'test_r2': test_r2
    })

    fitted_pipelines[name] = pipe

results = pd.DataFrame(rows).sort_values(by='test_rmse').reset_index(drop=True)
display(results)

In [ ]:
# Select final model based on the lowest test RMSE
best_model_name = results.loc[0, 'model']
final_model = fitted_pipelines[best_model_name]

final_pred = final_model.predict(X_test)
final_mae = mean_absolute_error(y_test, final_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, final_pred))
final_r2 = r2_score(y_test, final_pred)

print(f'Final model: {best_model_name}')
print(f'MAE  : {final_mae:.4f}')
print(f'RMSE : {final_rmse:.4f}')
print(f'R2   : {final_r2:.4f}')

In [ ]:
# Simple diagnostics: actual vs predicted and residual distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(x=y_test, y=final_pred, alpha=0.4, ax=axes[0])
axes[0].set_title('Actual vs Predicted')
axes[0].set_xlabel('Actual cnt')
axes[0].set_ylabel('Predicted cnt')

residuals = y_test - final_pred
sns.histplot(residuals, bins=40, kde=True, ax=axes[1])
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

### Conclusion
Kesimpulan diisi setelah seluruh cell dijalankan:
- Model final dipilih berdasarkan nilai RMSE test paling rendah.
- MAE digunakan untuk membaca rata-rata selisih prediksi terhadap nilai aktual.
- RMSE digunakan untuk memberi penalti lebih besar pada error besar.
- R-squared menunjukkan seberapa besar variasi `cnt` yang bisa dijelaskan model.

Template narasi:
- Model terbaik: **[isi otomatis dari output]**.
- Dengan MAE = **[nilai]** dan RMSE = **[nilai]**, model sudah/cukup baik untuk estimasi demand harian per observasi.
- Namun model tetap memiliki limitasi saat kondisi ekstrem/cuaca/event tertentu yang tidak tertangkap data.

### Recommendations
- Gunakan model untuk membantu perencanaan operasional (alokasi unit sepeda dan staffing) pada jam dengan demand tinggi.
- Lakukan retraining berkala agar model tetap relevan terhadap pola demand terbaru.
- Tambahkan fitur eksternal (misalnya event kota, kondisi cuaca detail, lokasi stasiun) untuk peningkatan performa.
- Terapkan monitoring performa model (drift data dan drift error) setelah implementasi.

### Save Model

In [ ]:
# Save final regression model and metadata
artifact = {
    'model_name': best_model_name,
    'pipeline': final_model,
    'feature_columns': feature_cols,
    'target_column': target_col,
    'metrics': {
        'mae': float(final_mae),
        'rmse': float(final_rmse),
        'r2': float(final_r2)
    }
}

with open('best_bike_sharing_regression_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print('Model saved to best_bike_sharing_regression_model.pkl')